# Sentiment Drift Paper — CPU-Only Track

**What this notebook covers (everything that doesn't need a GPU):**
- Phase 0: Environment + reproducibility scaffolding
- Phase 1: Full sarcasm baseline (VADER + RoBERTa on iSarcasm + SARC)
- Phase 2 (prep only): Load and cache HC3 + Yelp datasets, ready for the GPU notebook to consume directly -- no re-downloading needed later
- Phase 3 (partial): Stratified ECE for Bucket A (clear polarity) and Bucket B (sarcasm) -- both RoBERTa-based, both CPU-feasible
- Outputs everything to `/kaggle/working/results/` (or `/content/results/` on Colab) so the GPU notebook can pick up exactly where this leaves off

**What this notebook does NOT cover (needs the companion GPU notebook):**
- Qwen3.5 / Gemma 4 paraphrase generation (Phase 2 core)
- Semantic entropy sampling (Phase 3.5)
- Bucket C (AI-generated text) ECE -- depends on Phase 2's GPU output
- Explanation generation (Phase 5)

**Kaggle setup notes:**
- Create this as a Kaggle Notebook with **Accelerator: None** (CPU only) -- confirms you're not burning GPU quota by accident
- Enable "Internet" in notebook settings (needed for HuggingFace dataset downloads)
- Use "Save & Run All (Commit)" to run unattended -- Kaggle keeps CPU sessions running in the background even after you close the tab
- RoBERTa-base inference on ~15-20k examples (iSarcasm + SARC subset + TweetEval) takes roughly 30-90 minutes on Kaggle's CPU, depending on batch size -- leave it running and check back


## Phase 0 -- Environment Setup & Reproducibility Scaffolding (CPU-safe)

In [1]:
# --- Phase 0.1: Install pinned dependencies (CPU-only torch, smaller download) ---
!pip install -q \
    transformers==4.46.3 \
    datasets==3.1.0 \
    accelerate==1.1.1 \
    vaderSentiment==3.3.2 \
    scikit-learn==1.5.2 \
    scipy==1.13.1 \
    pandas==2.2.2 \
    numpy>=2.0.0 \
    matplotlib==3.9.2 \
    huggingface_hub==0.26.2

# NOTE: numpy pinned to >=2.0.0 (changed from ==1.26.4) -- the 1.26.4 pin caused a
# dependency-resolution conflict with other packages on current Colab/Kaggle base images.
# NOTE: torch is pre-installed on both Kaggle and Colab base images.
# We do NOT install bitsandbytes here -- that's GPU-only, needed in the companion notebook, not this one.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_versi

In [2]:
# --- Phase 0.2: Fixed seeds + platform detection ---
import random
import numpy as np
import torch
import os

SEED = 42

def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(SEED)

IS_KAGGLE = os.path.exists("/kaggle/working")
BASE_DIR = "/kaggle/working/results" if IS_KAGGLE else "/content/results"
print(f"Platform: {'Kaggle' if IS_KAGGLE else 'Colab/other'}")
print(f"Seed fixed to {SEED}. CUDA available: {torch.cuda.is_available()} (expected: False in this CPU notebook)")
print(f"Output base directory: {BASE_DIR}")


Platform: Kaggle
Seed fixed to 42. CUDA available: False (expected: False in this CPU notebook)
Output base directory: /kaggle/working/results


In [3]:
# --- Phase 0.3: Directory structure ---
import os

DIRS = {
    "data":      f"{BASE_DIR}/data",
    "phase1":    f"{BASE_DIR}/phase1_baseline",
    "phase2_prep": f"{BASE_DIR}/phase2_data_prep",
    "phase3":    f"{BASE_DIR}/phase3_calibration",
    "manifest":  f"{BASE_DIR}/manifest",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("Directory structure created under", BASE_DIR)
for k, v in DIRS.items():
    print(f"  {k:14s} -> {v}")


Directory structure created under /kaggle/working/results
  data           -> /kaggle/working/results/data
  phase1         -> /kaggle/working/results/phase1_baseline
  phase2_prep    -> /kaggle/working/results/phase2_data_prep
  phase3         -> /kaggle/working/results/phase3_calibration
  manifest       -> /kaggle/working/results/manifest


In [4]:
# --- Phase 0.4: Manifest accumulator ---
import sys, platform, json, hashlib
from datetime import datetime, timezone

MANIFEST = {
    "notebook": "cpu_track",
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "seed": SEED,
    "packages": {},
    "datasets": {},
    "models": {},
    "notes": [],
}

def log_package(name):
    try:
        mod = __import__(name)
        MANIFEST["packages"][name] = getattr(mod, "__version__", "unknown")
    except ImportError:
        MANIFEST["packages"][name] = "not importable"

def log_dataset(name, split, df_or_dataset, source_url=None, revision=None):
    n = len(df_or_dataset)
    try:
        sample_repr = str(df_or_dataset[:5]).encode("utf-8")
    except Exception:
        sample_repr = str(df_or_dataset).encode("utf-8")[:2000]
    h = hashlib.sha256(sample_repr).hexdigest()[:16]
    MANIFEST["datasets"][f"{name}::{split}"] = {
        "n_examples": n, "source_url": source_url, "revision": revision, "sample_hash": h,
    }

def log_model(name, revision=None, quantization=None):
    MANIFEST["models"][name] = {"revision": revision, "quantization": quantization}

for pkg in ["transformers", "datasets", "torch", "numpy", "pandas", "sklearn", "scipy"]:
    log_package(pkg)

print("Manifest logging initialized for CPU track.")


Manifest logging initialized for CPU track.


## Phase 1 -- Baseline Failure Rate (Sarcasm) -- Full Run, CPU-Safe

VADER and RoBERTa-base both run acceptably on CPU at this data volume. This is the entire
Phase 1 from the original plan, with no GPU dependency.


In [5]:
# --- Phase 1.1: Load iSarcasm ---
from datasets import load_dataset
import pandas as pd

try:
    isarcasm = load_dataset("tasksource/iSarcasm")
    isarcasm_df = isarcasm["train"].to_pandas() if "train" in isarcasm else isarcasm["test"].to_pandas()
    print("Loaded iSarcasm via tasksource mirror:", len(isarcasm_df), "rows")
except Exception as e:
    print("HF mirror failed, falling back to GitHub source:", e)
    import urllib.request
    url = "https://raw.githubusercontent.com/iabufarha/iSarcasmEval/main/train/train.En.csv"
    urllib.request.urlretrieve(url, f"{DIRS['data']}/isarcasm_train.csv")
    isarcasm_df = pd.read_csv(f"{DIRS['data']}/isarcasm_train.csv")
    print("Loaded iSarcasm via GitHub fallback:", len(isarcasm_df), "rows")

log_dataset("iSarcasm", "train", isarcasm_df, source_url="https://github.com/iabufarha/iSarcasmEval")
print(isarcasm_df.columns.tolist())
isarcasm_df.head()


HF mirror failed, falling back to GitHub source: Dataset 'tasksource/iSarcasm' doesn't exist on the Hub or cannot be accessed.
Loaded iSarcasm via GitHub fallback: 3468 rows
['Unnamed: 0', 'tweet', 'sarcastic', 'rephrase', 'sarcasm', 'irony', 'satire', 'understatement', 'overstatement', 'rhetorical_question']


,Unnamed: 0,tweet,sarcastic,rephrase,sarcasm,irony,satire,understatement,overstatement,rhetorical_question
0,0,The only thing I got from college is a caffein...,1,"College is really difficult, expensive, tiring...",0.0,1.0,0.0,0.0,0.0,0.0
1,1,I love it when professors draw a big question ...,1,I do not like when professors don’t write out ...,1.0,0.0,0.0,0.0,0.0,0.0
2,2,Remember the hundred emails from companies whe...,1,"I, at the bare minimum, wish companies actuall...",0.0,1.0,0.0,0.0,0.0,0.0
3,3,Today my pop-pop told me I was not “forced” to...,1,"Today my pop-pop told me I was not ""forced"" to...",1.0,0.0,0.0,0.0,0.0,0.0
4,4,@VolphanCarol @littlewhitty @mysticalmanatee I...,1,I would say Ted Cruz is an asshole and doesn’t...,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# --- Phase 1.2: Load SARC, stratified subset for adequate statistical power ---
# NOTE: dataset repo is "nikesh66/Sarcasm-dataset", not "nikesh66/SARC" -- the latter 404s.
# This mirror also uses a different column name/encoding than assumed originally
# ("Sarcasm (yes/no)" with string values) -- standardized below.
sarc = load_dataset("nikesh66/Sarcasm-dataset")
sarc_df = sarc["train"].to_pandas()

if "Sarcasm (yes/no)" in sarc_df.columns:
    sarc_df = sarc_df.rename(columns={"Sarcasm (yes/no)": "label"})
sarc_df["label"] = sarc_df["label"].map({"yes": 1, "no": 0})

print(f"SARC columns found: {sarc_df.columns.tolist()}")

N_PER_CLASS = 10000
sarc_balanced = (
    sarc_df.groupby("label", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), N_PER_CLASS), random_state=SEED))
    .reset_index(drop=True)
)

log_dataset("SARC", "balanced_eval_subset", sarc_balanced,
            source_url="https://huggingface.co/datasets/nikesh66/Sarcasm-dataset",
            revision=f"stratified_{N_PER_CLASS}_per_class_seed{SEED}")
print(f"SARC balanced eval subset: {len(sarc_balanced)} rows "
      f"({sarc_balanced['label'].value_counts().to_dict()})")


In [ ]:
# --- Phase 1.3: Load TweetEval (also needed for Phase 3 Bucket A later in this notebook) ---
tweeteval = load_dataset("tweet_eval", "sentiment")
tweeteval_test = tweeteval["test"].to_pandas()

log_dataset("TweetEval", "test", tweeteval_test,
            source_url="https://huggingface.co/datasets/tweet_eval", revision="sentiment_config")
print(f"TweetEval test set: {len(tweeteval_test)} rows")


In [ ]:
# --- Phase 1.4: VADER baseline ---
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def vader_predict(text):
    scores = analyzer.polarity_scores(str(text))
    compound = scores["compound"]
    if compound >= 0.05:
        return "positive", min(1.0, abs(compound) + 0.5)
    elif compound <= -0.05:
        return "negative", min(1.0, abs(compound) + 0.5)
    return "neutral", 1.0 - abs(compound)

text_col = "tweet" if "tweet" in isarcasm_df.columns else isarcasm_df.columns[0]
isarcasm_df["vader_label"], isarcasm_df["vader_conf"] = zip(*isarcasm_df[text_col].apply(vader_predict))
print("VADER predictions complete on iSarcasm.")


In [ ]:
# --- Phase 1.5: RoBERTa baseline (CPU inference -- slower than GPU but entirely feasible) ---
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
import time

# IMPORTANT: force the PyTorch backend explicitly. A prior run on this exact model name
# silently loaded the TensorFlow backend (TFRobertaForSequenceClassification) instead,
# which left the classifier head randomly initialized (transformers printed a warning:
# "Some layers ... were not initialized from the model checkpoint ... ['classifier']" --
# easy to miss in a long log). The result was a broken model that predicted "neutral" for
# 3466/3468 rows -- not a real finding, just an uninitialized classifier outputting near-
# constant garbage. Loading the model object directly via from_pretrained (rather than
# letting `pipeline()` auto-pick a framework) avoids this ambiguity.

roberta_model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name, revision="main")
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, revision="main")

roberta_pipe = pipeline(
    "sentiment-analysis",
    model=roberta_model,
    tokenizer=roberta_tokenizer,
    framework="pt",   # explicit -- do not let this silently fall back to tf
    device=-1,
    truncation=True,
)
log_model(roberta_model_name, revision="main")

texts = isarcasm_df[text_col].astype(str).tolist()
BATCH = 16
start = time.time()
roberta_preds = []
for i in range(0, len(texts), BATCH):
    roberta_preds.extend(roberta_pipe(texts[i:i+BATCH]))
    if i % (BATCH * 20) == 0:
        elapsed = time.time() - start
        print(f"  {i}/{len(texts)} done, {elapsed:.0f}s elapsed")

isarcasm_df["roberta_label"] = [p["label"].lower() for p in roberta_preds]
isarcasm_df["roberta_conf"] = [p["score"] for p in roberta_preds]
print(f"RoBERTa predictions complete on iSarcasm in {time.time()-start:.0f}s total.")

# --- Sanity check: catch a broken/randomly-initialized classifier head before it
# silently poisons every downstream number. A healthy 3-class sentiment model on
# ~3,400 mixed tweets should NOT predict the same label for >90% of rows.
label_counts = isarcasm_df["roberta_label"].value_counts(normalize=True)
max_label_share = label_counts.max()
if max_label_share > 0.90:
    raise RuntimeError(
        f"SANITY CHECK FAILED: {max_label_share:.1%} of predictions are the same label "
        f"({label_counts.idxmax()!r}). This matches the signature of a randomly-initialized "
        f"classifier head (see the cell above's note) rather than a real sentiment model. "
        f"Check the cell output above for a warning like 'Some layers ... were not "
        f"initialized from the model checkpoint ... [classifier]' before re-running -- "
        f"do NOT trust any Phase 1/3 numbers until this is resolved."
    )
print(f"Sanity check passed: label distribution looks like a working classifier "
      f"(max single-label share: {max_label_share:.1%}).")


In [ ]:
# --- Phase 1.6: Resolve label schema and compute failure-rate metrics ---
from sklearn.metrics import f1_score, accuracy_score
import json

label_col = "sarcastic" if "sarcastic" in isarcasm_df.columns else isarcasm_df.columns[-1]
print("Using label column:", label_col, "| unique values:", isarcasm_df[label_col].unique()[:5])

SARCASM_POSITIVE_VALUES = {1, "1", True, "sarcastic"}
isarcasm_df["is_sarcastic"] = isarcasm_df[label_col].apply(lambda x: x in SARCASM_POSITIVE_VALUES)

results_phase1 = {
    "n_isarcasm": len(isarcasm_df),
    "n_sarc_balanced": len(sarc_balanced),
    "n_tweeteval": len(tweeteval_test),
    "vader_label_distribution": isarcasm_df["vader_label"].value_counts().to_dict(),
    "roberta_label_distribution": isarcasm_df["roberta_label"].value_counts().to_dict(),
    "pct_sarcastic_in_isarcasm": round(isarcasm_df["is_sarcastic"].mean(), 4),
}

with open(f"{DIRS['phase1']}/baseline_results.json", "w") as f:
    json.dump(results_phase1, f, indent=2)

print(json.dumps(results_phase1, indent=2))


## Phase 2 (Prep Only) -- Cache HC3 and Yelp Now, So the GPU Notebook Doesn't Re-Download

This is purely I/O -- downloading and stratified-sampling the datasets the GPU notebook needs.
Doing this now means your 5-days-from-now GPU session starts generating immediately instead of
spending its first 10-15 minutes on dataset downloads.


In [ ]:
# --- Phase 2.1 (prep): Load and cache HC3 ---
# IMPORTANT: load_dataset("Hello-SimpleAI/HC3", "all") fails on current `datasets` versions
# (>=4.0.0 dropped support for script-based dataset loaders entirely, and HC3's repo still
# ships a legacy HC3.py loader script -- "RuntimeError: Dataset scripts are no longer
# supported, but found HC3.py"). Pinning datasets==3.1.0 in Phase 0.1 SHOULD avoid this,
# but Kaggle's base image can override/upgrade it regardless -- don't rely on the pin alone.
#
# Robust fix: bypass load_dataset entirely and pull the raw all.jsonl file directly via
# hf_hub_download. This works regardless of `datasets` version since it doesn't touch the
# script-loading path at all.

from huggingface_hub import hf_hub_download
import json as _json

hc3_jsonl_path = hf_hub_download(repo_id="Hello-SimpleAI/HC3", filename="all.jsonl", repo_type="dataset")

hc3_records = []
with open(hc3_jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            hc3_records.append(_json.loads(line))

hc3_df = pd.DataFrame(hc3_records)
hc3_df.to_parquet(f"{DIRS['phase2_prep']}/hc3_full.parquet")

log_dataset("HC3", "train", hc3_df, source_url="https://huggingface.co/datasets/Hello-SimpleAI/HC3")
print(f"HC3 cached: {len(hc3_df)} rows -> {DIRS['phase2_prep']}/hc3_full.parquet")
print(f"HC3 columns: {hc3_df.columns.tolist()}")


In [ ]:
# --- Phase 2.2 (prep): Load Yelp Polarity, draw stratified sample, cache ---
yelp = load_dataset("yelp_polarity")
yelp_test = yelp["test"].to_pandas()

N_YELP = 5000
yelp_sample = (
    yelp_test.groupby("label", group_keys=False)
    .apply(lambda x: x.sample(N_YELP // 2, random_state=SEED))
    .reset_index(drop=True)
)
yelp_sample.to_parquet(f"{DIRS['phase2_prep']}/yelp_sample.parquet")

log_dataset("YelpPolarity", "stratified_sample", yelp_sample,
            source_url="https://huggingface.co/datasets/yelp_polarity",
            revision=f"n{N_YELP}_seed{SEED}")
print(f"Yelp stratified sample cached: {len(yelp_sample)} rows -> {DIRS['phase2_prep']}/yelp_sample.parquet")


In [ ]:
# --- Phase 2.3 (prep): Filter HC3 for sentiment-bearing pairs (second drift domain) ---
OPINION_KEYWORDS = [
    "think", "feel", "opinion", "believe", "recommend", "love", "hate",
    "best", "worst", "favorite", "review", "like", "dislike", "enjoy",
]

def is_opinion_bearing(question):
    q = str(question).lower()
    return any(kw in q for kw in OPINION_KEYWORDS)

hc3_df["likely_opinion_bearing"] = hc3_df["question"].apply(is_opinion_bearing) if "question" in hc3_df.columns else False
hc3_opinion_subset = hc3_df[hc3_df["likely_opinion_bearing"]].copy()
hc3_opinion_subset.to_parquet(f"{DIRS['phase2_prep']}/hc3_opinion_subset.parquet")

print(f"HC3 opinion-bearing subset: {len(hc3_opinion_subset)} / {len(hc3_df)} rows "
      f"({100*len(hc3_opinion_subset)/max(len(hc3_df),1):.1f}%)")
print(">>> If this subset is too small (<500 rows) for a second-domain drift experiment,")
print(">>> the GPU notebook should fall back to the Amazon Reviews second-domain plan")
print(">>> noted in the closing gaps section instead.")


## Phase 3 (Partial) -- Stratified ECE for Buckets A and B

Bucket C (AI-generated text) depends on the GPU notebook's paraphrase output and is computed
there. Buckets A and B only need RoBERTa, which we already ran on CPU above.


In [ ]:
# --- Phase 3.1: ECE utility ---
def expected_calibration_error(confidences, correctness, n_bins=10):
    confidences = np.asarray(confidences)
    correctness = np.asarray(correctness)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_stats = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        mask = (confidences >= lo) & (confidences < hi) if i < n_bins - 1 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            bin_stats.append({"bin": (round(lo,2), round(hi,2)), "n": 0, "acc": None, "conf": None})
            continue
        bin_acc = correctness[mask].mean()
        bin_conf = confidences[mask].mean()
        bin_weight = mask.sum() / len(confidences)
        ece += bin_weight * abs(bin_acc - bin_conf)
        bin_stats.append({"bin": (round(lo,2), round(hi,2)), "n": int(mask.sum()),
                           "acc": round(float(bin_acc),4), "conf": round(float(bin_conf),4)})
    return ece, bin_stats


In [ ]:
# --- Phase 3.2: Bucket A (clear polarity, TweetEval) ---
te_subset_texts = tweeteval_test["text"].astype(str).tolist()[:2000]
te_preds = []
for i in range(0, len(te_subset_texts), BATCH):
    te_preds.extend(roberta_pipe(te_subset_texts[i:i+BATCH]))

tweeteval_subset = tweeteval_test.iloc[:2000].copy()
tweeteval_subset["roberta_label"] = [p["label"].lower() for p in te_preds]
tweeteval_subset["roberta_conf"] = [p["score"] for p in te_preds]

te_label_map = {0: "negative", 1: "neutral", 2: "positive"}
tweeteval_subset["true_label"] = tweeteval_subset["label"].map(te_label_map)
tweeteval_subset["correct"] = (tweeteval_subset["roberta_label"] == tweeteval_subset["true_label"]).astype(int)

ece_clear, bins_clear = expected_calibration_error(tweeteval_subset["roberta_conf"], tweeteval_subset["correct"])
print(f"Bucket A (clear polarity) ECE: {ece_clear:.4f}")


In [ ]:
# --- Phase 3.3: Bucket B (sarcasm, iSarcasm) ---
# CONFIRMED from the actual run output: iSarcasm's columns are
# ['Unnamed: 0', 'tweet', 'sarcastic', 'rephrase', 'sarcasm', 'irony', 'satire',
#  'understatement', 'overstatement', 'rhetorical_question'] -- there is NO sentiment
# polarity ground-truth column. The earlier placeholder (mapping is_sarcastic to an
# assumed negative/positive label) was a fabricated proxy, not real ground truth, and
# produced an inflated, untrustworthy ECE (0.38 vs Bucket A's 0.056 -- too clean a gap
# to trust). Replacing it with a metric Bucket B can actually support: PREDICTION
# INSTABILITY, not accuracy-based ECE.
#
# Rationale: without a sentiment label we can't ask "was the model's confidence
# calibrated to correctness." We CAN ask "is the model's confidence systematically
# lower / more erratic on sarcastic text than non-sarcastic text" -- this is a
# legitimate, defensible claim that doesn't require fabricating ground truth.

sarcastic_conf = isarcasm_df.loc[isarcasm_df["is_sarcastic"], "roberta_conf"]
non_sarcastic_conf = isarcasm_df.loc[~isarcasm_df["is_sarcastic"], "roberta_conf"]

from scipy.stats import mannwhitneyu

instability_stats = {
    "mean_confidence_sarcastic": round(float(sarcastic_conf.mean()), 4),
    "mean_confidence_non_sarcastic": round(float(non_sarcastic_conf.mean()), 4),
    "std_confidence_sarcastic": round(float(sarcastic_conf.std()), 4),
    "std_confidence_non_sarcastic": round(float(non_sarcastic_conf.std()), 4),
    "n_sarcastic": int(isarcasm_df["is_sarcastic"].sum()),
    "n_non_sarcastic": int((~isarcasm_df["is_sarcastic"]).sum()),
}

# Mann-Whitney U test: is the confidence distribution on sarcastic text significantly
# different (typically lower / more spread) than on non-sarcastic text?
u_stat, p_value = mannwhitneyu(sarcastic_conf, non_sarcastic_conf, alternative="two-sided")
instability_stats["mannwhitney_u"] = round(float(u_stat), 2)
instability_stats["mannwhitney_p"] = round(float(p_value), 6)

print("Bucket B (sarcasm) -- confidence instability, not ECE (no sentiment ground truth available):")
print(json.dumps(instability_stats, indent=2))
print(">>> If mannwhitney_p < 0.05, this supports 'RoBERTa is measurably less confident/more")
print(">>> erratic on sarcastic text' as your Bucket B finding -- a real claim, not a fabricated ECE.")
print(">>> If you still want a true Bucket B ECE for the paper, that requires sourcing a")
print(">>> SEPARATE iSarcasm release/split that includes sentiment polarity annotations, or")
print(">>> using SemEval iSarcasmEval's task B sentiment labels if available -- check before writing.")

# Set ece_sarcasm to None downstream rather than reporting a number we know is unsound.
ece_sarcasm = None
bins_sarcasm = None


In [ ]:
# --- Phase 3.4: Save partial stratified results + reliability diagram (Bucket A only) ---
import matplotlib.pyplot as plt
import json

partial_ece_results = {
    "clear_polarity_ECE": round(ece_clear, 4),
    "sarcasm_ECE": None,  # no longer reported as ECE -- see Bucket B instability stats instead
    "sarcasm_instability_stats": instability_stats,
    "ai_generated_text_ECE": None,
}
with open(f"{DIRS['phase3']}/stratified_ece_partial.json", "w") as f:
    json.dump(partial_ece_results, f, indent=2)

def plot_reliability(bin_stats_list, labels, save_path):
    fig, axes = plt.subplots(1, len(bin_stats_list), figsize=(5*len(bin_stats_list), 4.5), sharey=True)
    if len(bin_stats_list) == 1:
        axes = [axes]
    for ax, bins, label in zip(axes, bin_stats_list, labels):
        confs = [b["conf"] for b in bins if b["conf"] is not None]
        accs  = [b["acc"]  for b in bins if b["acc"]  is not None]
        ax.plot([0,1],[0,1], linestyle="--", color="gray", label="Perfect calibration")
        ax.bar(confs, accs, width=0.08, alpha=0.7, edgecolor="black", label="Observed")
        ax.set_title(label); ax.set_xlabel("Confidence"); ax.set_xlim(0,1); ax.set_ylim(0,1)
    axes[0].set_ylabel("Accuracy"); axes[0].legend(loc="upper left", fontsize=8)
    plt.tight_layout(); plt.savefig(save_path, dpi=200); plt.show()

plot_reliability(
    [bins_clear],
    ["Clear Polarity (TweetEval) -- Bucket A"],
    f"{DIRS['phase3']}/reliability_diagram_bucket_a.png"
)

# Bucket B gets a different plot type -- confidence distributions, not a reliability
# diagram, since it has no correctness/ECE basis (see Phase 3.3 note).
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.hist(sarcastic_conf, bins=20, alpha=0.6, label=f"Sarcastic (n={instability_stats['n_sarcastic']})", density=True)
ax.hist(non_sarcastic_conf, bins=20, alpha=0.6, label=f"Non-sarcastic (n={instability_stats['n_non_sarcastic']})", density=True)
ax.set_xlabel("RoBERTa confidence"); ax.set_ylabel("Density")
ax.set_title(f"Bucket B: Confidence distribution (Mann-Whitney p={instability_stats['mannwhitney_p']:.4g})")
ax.legend()
plt.tight_layout()
plt.savefig(f"{DIRS['phase3']}/confidence_distribution_bucket_b.png", dpi=200)
plt.show()

print(json.dumps(partial_ece_results, indent=2))


## Manifest + Handoff to the GPU Notebook

In [ ]:
# --- Final manifest for this CPU run ---
MANIFEST["notes"].append("iSarcasm label_col CONFIRMED via actual run: column is 'sarcastic', binary 1/0, 25% positive class. Resolved.")
MANIFEST["notes"].append("Bucket B redefined from ECE to confidence-instability (Mann-Whitney) since iSarcasm has no sentiment-polarity ground-truth column -- confirmed from actual schema, not assumed.")
MANIFEST["notes"].append("SARC license terms must be verified before redistribution.")
MANIFEST["notes"].append(f"Cached for GPU notebook: hc3_full.parquet, yelp_sample.parquet, hc3_opinion_subset.parquet in {DIRS['phase2_prep']}")

manifest_path = f"{DIRS['manifest']}/manifest_cpu_track.json"
with open(manifest_path, "w") as f:
    json.dump(MANIFEST, f, indent=2)

print(f"Manifest written to {manifest_path}")
print("\n" + "="*70)
print("CPU TRACK COMPLETE.")
print("="*70)
print(f'''
Outputs ready for the GPU notebook, all under {BASE_DIR}:
  - {DIRS["phase2_prep"]}/hc3_full.parquet
  - {DIRS["phase2_prep"]}/yelp_sample.parquet
  - {DIRS["phase2_prep"]}/hc3_opinion_subset.parquet
  - {DIRS["phase1"]}/baseline_results.json
  - {DIRS["phase3"]}/stratified_ece_partial.json

NEXT STEP: when your Kaggle GPU quota resets, persist these output files via
Kaggle's "Add Data" -> "Notebook Output Files" feature so the GPU notebook can
attach this notebook's output as a data source and start generating immediately,
without re-running any CPU work.
''')
